# Marimba quickstart on Google Colab

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Charcot-AI/marimba-api/blob/main/notebooks/colab_quickstart.ipynb)

This notebook installs the published `marimba-audio` 0.1.0 package and runs its real model. It uses an eight-second synthetic 440 Hz tone-and-silence clip generated below, not a recording of speech. The resulting decisions demonstrate the API; they are not accuracy measurements.

The PyPI `torch` extra pins Torch and TorchAudio to 2.8.0. Colab's CUDA build with that base version satisfies the pins; the device check below reports whether CUDA remains available. A Hugging Face token is only needed while the model weights are private. In Colab, add it as a notebook secret named `HF_TOKEN`; never paste it into a cell.

In [1]:
!pip install -q "marimba-audio[torch]==0.1.0"


[notice] A new release of pip is available: 25.0.1 -> 26.3
[notice] To update, run: pip install --upgrade pip


In [2]:
import os
import time
import warnings

import numpy as np
import torch

from marimba import Marimba, Question

warnings.filterwarnings("ignore", message="IProgress not found.*")
warnings.filterwarnings("ignore", message="enable_nested_tensor is True.*")

try:
    from google.colab import userdata

    hf_token = userdata.get("HF_TOKEN")
except Exception:
    hf_token = os.environ.get("HF_TOKEN")

if hf_token:
    os.environ["HF_TOKEN"] = hf_token
    print("Hugging Face token available; value is not displayed.")
else:
    print("No HF_TOKEN found. A token is only needed while the model weights are private.")

print(f"PyTorch {torch.__version__}; CUDA runtime: {torch.version.cuda}")
if torch.cuda.is_available():
    device = "cuda"
    print(f"Using GPU: {torch.cuda.get_device_name(0)}")
else:
    device = "cpu"
    print("CUDA is unavailable; falling back to CPU.")

if device == "cpu":
    torch.set_num_threads(4)

sample_rate = 16_000
duration_s = 8
time_s = np.arange(sample_rate * duration_s, dtype=np.float32) / sample_rate
audio = np.zeros(sample_rate * duration_s, dtype=np.float32)
for start_s, end_s in ((0.5, 2.0), (3.5, 5.0), (6.0, 7.0)):
    start = int(start_s * sample_rate)
    end = int(end_s * sample_rate)
    audio[start:end] = 0.1 * np.sin(2 * np.pi * 440 * time_s[start:end])
print("Synthetic audio: 440 Hz tone bursts separated by silence; no speech recording.")

model = Marimba.from_pretrained(backend="torch", device=device, dtype="float32")

No HF_TOKEN found. A token is only needed while the model weights are private.
PyTorch 2.8.0+cpu; CUDA runtime: None
CUDA is unavailable; falling back to CPU.
Synthetic audio: 440 Hz tone bursts separated by silence; no speech recording.


Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

Fetching 10 files: 100%|██████████| 10/10 [00:00<00:00, 3207.14it/s]

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


In [3]:
questions = [
    Question.noul("Is someone speaking?"),
    Question.score("Has the speaker finished?"),
    Question.choice(
        "What language is being spoken?", ["English", "Portuguese", "Spanish"], task="language"
    ),
    Question.noul('Did someone say "marimba"?'),
    Question.score("Did the speaker change?"),
    Question.choice("How many people are talking?", ["0", "1", "2", "3+"]),
]

started = time.perf_counter()
decisions = model(audio, questions, sampling_rate=sample_rate)
call_latency_ms = (time.perf_counter() - started) * 1000
print({"device": device, "call_latency_ms": round(call_latency_ms, 2)})
for decision in decisions:
    print(decision.to_dict())

Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


{'device': 'cpu', 'call_latency_ms': 2098.88}
{'question': {'text': 'Is someone speaking?', 'kind': 'noul', 'options': None, 'task': 'voice_activity', 'threshold': None, 'seconds': None}, 'task': 'voice_activity', 'kind': 'noul', 'value': False, 'probability': 1.7188386538783676e-05, 'probabilities': None, 'score': -7.175867080688477, 'threshold': 0.225, 'window_start_s': 0.0, 'window_end_s': 8.0, 'latency_ms': 2098.491085926071, 'status': 'validated', 'calibrated': False}
{'question': {'text': 'Has the speaker finished?', 'kind': 'score', 'options': None, 'task': 'turn_end', 'threshold': None, 'seconds': None}, 'task': 'turn_end', 'kind': 'score', 'value': 0.9447765591117404, 'probability': 0.9447765591117404, 'probabilities': None, 'score': 0.9421308514324253, 'threshold': 0.5, 'window_start_s': 0.0, 'window_end_s': 8.0, 'latency_ms': 2098.491085926071, 'status': 'validated', 'calibrated': False}
{'question': {'text': 'What language is being spoken?', 'kind': 'choice', 'options': ['E